# VAZHI GGUF v7.1 — Quality Validation

**Lineage:** `google/gemma-3-1b-it` → SFT v7.0 (LoRA r=8) → SFT v7.1 (LoRA r=16) → **GGUF Q4_K_M**

**GGUF conversion** was done via [ggml-org/gguf-my-repo](https://huggingface.co/spaces/ggml-org/gguf-my-repo) Space.

**GGUF repo:** [`CryptoYogi/vazhi-v7_1-Q4_K_M-GGUF`](https://huggingface.co/CryptoYogi/vazhi-v7_1-Q4_K_M-GGUF)

**This notebook validates** the Q4_K_M GGUF output quality before mobile deployment:
1. Download the GGUF file
2. Run 5 Tamil prompts — GO/NO-GO gate
3. Run 10-prompt full eval across all domains
4. Compute SHA256 checksum for app integrity verification

**Runtime:** Colab Pro, CPU sufficient (no GPU needed for llama.cpp CPU inference)

In [ ]:
# Cell 1 — Dependencies + llama.cpp Build
!pip install -q huggingface_hub

# Clone llama.cpp (cmake build required for Gemma 3 architecture support)
!git clone https://github.com/ggerganov/llama.cpp.git

# Build with cmake (not make — needed for Gemma 3)
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release -j4
%cd ..

print("\n" + "="*50)
print("llama.cpp built successfully")
!ls -la llama.cpp/build/bin/llama-cli

In [ ]:
# Cell 2 — Configuration

GGUF_REPO = "CryptoYogi/vazhi-v7_1-Q4_K_M-GGUF"
GGUF_FILE = "vazhi-v7_1-q4_k_m.gguf"  # 806 MB

# Gemma 3 chat template (no system role — embed identity in user turn)
SYSTEM_CONTEXT = "நீங்கள் வழி (VAZHI), தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர்."

print(f"GGUF repo: {GGUF_REPO}")
print(f"GGUF file: {GGUF_FILE}")

In [ ]:
# Cell 3 — Download GGUF from HuggingFace
from huggingface_hub import hf_hub_download
import os

print(f"Downloading {GGUF_FILE} from {GGUF_REPO}...")
local_path = hf_hub_download(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
    local_dir=".",
)

size_bytes = os.path.getsize(GGUF_FILE)
print(f"\nDownloaded: {GGUF_FILE}")
print(f"Size: {size_bytes:,} bytes ({size_bytes / 1e6:.1f} MB)")
print(f"Within <1GB limit: {'PASS' if size_bytes < 1e9 else 'FAIL'}")

In [ ]:
# Cell 4 — GO/NO-GO Gate: 5 Tamil Prompts
#
# Gemma 3 prompt format: no system role, embed identity in user turn.

import subprocess

TEST_PROMPTS = [
    ("greeting",  "வணக்கம்"),
    ("identity",  "நீங்கள் யார்?"),
    ("factual",   "தமிழ்நாட்டின் தலைநகரம் எது?"),
    ("domain",    "முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்"),
    ("culture",   "திருக்குறள் பற்றி சொல்லுங்கள்"),
]

def build_gemma_prompt(user_msg):
    """Build Gemma 3 prompt with VAZHI identity embedded in user turn."""
    return (
        f"<start_of_turn>user\n"
        f"{SYSTEM_CONTEXT}\n\n"
        f"{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

def run_inference(gguf_path, prompt_text, max_tokens=150):
    """Run llama-cli inference and return generated text."""
    cmd = [
        "./llama.cpp/build/bin/llama-cli",
        "-m", gguf_path,
        "-p", prompt_text,
        "-n", str(max_tokens),
        "--temp", "0.7",
        "-ngl", "0",
        "--no-display-prompt",
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        output = result.stdout.strip()
        # Remove any trailing special tokens
        for tok in ["<end_of_turn>", "<eos>"]:
            output = output.split(tok)[0]
        return output.strip()
    except subprocess.TimeoutExpired:
        return "[TIMEOUT]"
    except Exception as e:
        return f"[ERROR: {e}]"

# Run GO/NO-GO tests
print("GO/NO-GO GATE — 5 Tamil Prompts")
print("="*80)

results = []
for category, prompt in TEST_PROMPTS:
    full_prompt = build_gemma_prompt(prompt)
    output = run_inference(GGUF_FILE, full_prompt)
    results.append((category, prompt, output))

    print(f"\n[{category}] {prompt}")
    print(f"{'─'*60}")
    display = output[:400] + "..." if len(output) > 400 else output
    print(display)

# GO/NO-GO verdict
print("\n" + "="*80)
non_empty = sum(1 for _, _, o in results if len(o.strip()) > 10)
print(f"Q4_K_M produced {non_empty}/5 non-trivial responses.")
if non_empty >= 3:
    print("GO — Q4_K_M produces coherent output. Proceed to full eval.")
else:
    print("NO-GO — Q4_K_M output is garbage/empty. Investigate before proceeding.")

In [ ]:
# Cell 5 — Full Eval: Q4_K_M with 10 Prompts

FULL_EVAL_PROMPTS = [
    ("greeting",    "வணக்கம், எப்படி இருக்கிறீர்கள்?"),
    ("identity",    "நீங்கள் யார்? உங்களை பற்றி சொல்லுங்கள்"),
    ("factual",     "தமிழ்நாட்டின் தலைநகரம் எது?"),
    ("security",    "யாரோ என் வங்கி கணக்கு விவரங்கள் கேட்கிறார்கள், என்ன செய்வது?"),
    ("govt",        "முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்"),
    ("education",   "தமிழ்நாடு அரசு உதவித்தொகை பற்றி சொல்லுங்கள்"),
    ("health",      "சர்க்கரை நோய் தடுப்பது எப்படி?"),
    ("culture",     "திருக்குறள் பற்றி சொல்லுங்கள்"),
    ("legal",       "RTI மனு எப்படி போடுவது?"),
    ("spiritual",   "மன அமைதி எப்படி பெறுவது?"),
]

print("Full evaluation of Q4_K_M with 10 prompts...\n")
print("="*80)

eval_results = []
for category, prompt in FULL_EVAL_PROMPTS:
    full_prompt = build_gemma_prompt(prompt)
    output = run_inference(GGUF_FILE, full_prompt, max_tokens=200)
    eval_results.append((category, prompt, output))

    print(f"\n[{category}] {prompt}")
    print(f"{'─'*60}")
    display = output[:500] + "..." if len(output) > 500 else output
    print(display)

# Summary
print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)

for category, prompt, output in eval_results:
    length = len(output.strip())
    is_empty = length < 10
    is_repetitive = False
    if length > 50:
        chunk = output[20:40]
        if chunk and output.count(chunk) > 3:
            is_repetitive = True

    status = "EMPTY" if is_empty else ("REPETITIVE" if is_repetitive else "OK")
    print(f"  [{category:12s}] {status:10s} ({length:4d} chars)")

ok_count = sum(1 for _, _, o in eval_results if len(o.strip()) >= 10)
print(f"\nResult: {ok_count}/10 non-trivial responses")

In [ ]:
# Cell 6 — SHA256 Checksum + App Integration Summary
import os
import hashlib

print("Computing SHA256 checksum...")
sha256 = hashlib.sha256()
with open(GGUF_FILE, "rb") as f:
    while True:
        chunk = f.read(8192)
        if not chunk:
            break
        sha256.update(chunk)
checksum = sha256.hexdigest()

q4_size = os.path.getsize(GGUF_FILE)
download_url = f"https://huggingface.co/{GGUF_REPO}/resolve/main/{GGUF_FILE}"

print("\n" + "="*70)
print("VAZHI GGUF v7.1 — APP INTEGRATION SUMMARY")
print("="*70)

print(f"\n1. GGUF FILE")
print(f"   File:     {GGUF_FILE}")
print(f"   Size:     {q4_size:,} bytes ({q4_size/1e6:.1f} MB)")
print(f"   SHA256:   {checksum}")
print(f"   Limit:    {'PASS (<1GB)' if q4_size < 1e9 else 'FAIL (>1GB)'}")

print(f"\n2. DOWNLOAD URL")
print(f"   {download_url}")

print(f"\n3. SYSTEM PROMPT FORMAT (Gemma 3 — no system role)")
print(f"   Embed identity in user turn:")
print(f"   <start_of_turn>user")
print(f"   {SYSTEM_CONTEXT}")
print(f"   ")
print(f"   {{user_message}}<end_of_turn>")
print(f"   <start_of_turn>model")

print(f"\n4. FLUTTER APP VALUES (already updated)")
print(f"   modelUrl         = '{download_url}'")
print(f"   modelFilename    = '{GGUF_FILE}'")
print(f"   expectedModelSize = {q4_size}")
print(f"   expectedSha256   = '{checksum}'")

print(f"\n" + "="*70)